In [1]:
!pip install slices numba -q

In [2]:
!cd .. && ./apply_patch.sh && cd -

/home/shtnp/Crystal-generator-using-SLICES/notebooks


In [3]:
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
sys.path.append('..')
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore", message=".*You are casting an input of type complex64.*")

In [ ]:
from pymatgen.core import Structure
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

def compare_structures(s1, s2, to_print=False):
    # s1 = original_structure
    # s2 = backend.SLICES2structure(slices_NdSiRu)[0]

    ltol=0.3
    stol=0.5
    angle_tol=5

    tol = ltol
    for a, b in zip(s1.lattice.abc, s2.lattice.abc):
        if abs(a-b)/max(a,b) > tol:
            if to_print: print(f"Lattice lengths differ by more than {int(ltol * 100)}%:", s1.lattice.abc, s2.lattice.abc)
            return False
    for α, β in zip(s1.lattice.angles, s2.lattice.angles):
        if abs(α-β) > angle_tol:
            if to_print: print("Lattice angles differ by more than {angle_tol}°:", s1.lattice.angles, s2.lattice.angles)
            return False

    # spg1 = SpacegroupAnalyzer(s1).get_space_group_symbol()
    # spg2 = SpacegroupAnalyzer(s2).get_space_group_symbol()
    # if spg1 != spg2:
    #     print(f"Different space groups: {spg1} vs {spg2}")
    # else:
    #     print(f"Same space group: {spg1}")

    matcher = StructureMatcher(
        ltol=ltol,
        stol=stol,
        angle_tol=angle_tol,
        primitive_cell=True,
        scale=False
    )

    are_fit = matcher.fit(s1.get_primitive_structure(), s2.get_primitive_structure())

    if to_print:
        if are_fit:
            print("Structures match (within tolerances) 🎉")
        else:
            print("Structures do not match.")
    return are_fit

In [5]:
import re

def get_materail_ids(file):
    rows = []
    pattern = r'^(?P<status>E|x)?\s*(?P<idx>\d+):\s*(?P<material>[a-zA-Z]+-\d+)\s*---\s*(?P<time>[\d\.eE+-]+)'

    for line in open(file).read().strip().split('\n'):
        m = re.match(pattern, line)
        if m:
            rows.append(m.group('material'))
    return rows

In [ ]:
import multiprocessing as mp
import psutil
import time
import os

def run_with_memory_limit(target, mem_frac=0.3, poll_interval=0.5, *args, **kwargs):
    ctx = mp.get_context('spawn')
    queue = ctx.Queue()
    proc = ctx.Process(target=target, args=(queue, *args), kwargs=kwargs)
    proc.start()

    ps_proc = psutil.Process(proc.pid)
    total_mem = psutil.virtual_memory().total

    result = None
    killed = False

    while proc.is_alive():
        try:
            mem = ps_proc.memory_info().rss
            if mem / total_mem > mem_frac:
                print(f"Killing process {proc.pid}: using {mem/total_mem:.2%} RAM")
                proc.terminate()
                killed = True
                break
        except psutil.NoSuchProcess:
            break
        time.sleep(poll_interval)

    if not killed and not queue.empty():
        result = queue.get()
    proc.join()
    if killed:
        return Exception("Memory exceeded")
    return result


In [ ]:
from src.slices_pipeline.encoder import *
from src.slices_pipeline.loader import *
from slices.core import SLICES
from pymatgen.core.structure import Structure
from multiprocessing import Process, Queue
from pymatgen.io.cif import CifParser
from io import StringIO
from tqdm import tqdm
from queue import Empty
import time
import os
import gc
import psutil

import logging
import tensorflow as tf

ram_threshold = 50

# tf.get_logger().setLevel(logging.ERROR)

def get_mp20_split(split):
    if split is None:
        df = pd.concat([read_split_df("train"), read_split_df("test"), read_split_df("val")])
        return df, "full"
    else:
        df = read_split_df(split)
        return df, split

def job(rows, queue, worker_id, free_queue):
    # tf.get_logger().setLevel(logging.ERROR)

    # sys.stdout = open(os.devnull, 'w')
    # sys.stderr = open(os.devnull, 'w')

    def get_relax(queue, slices):
        queue.put(backend.SLICES2structure(slices_NdSiRu))

    backend = SLICES(relax_model="chgnet", steps=100)
    # backend = SLICES()

    gc_each = 10
    i = 0

    for idx, row in rows.iterrows():
        start_time = time.time()
        material_id = row.material_id
        try:
            # original_structure = Structure.from_str(cif, fmt="cif", primitive=True)
            cif = row.cif
            original_structure = CifParser(StringIO(cif)).parse_structures()[0]
            # print(i, df.iloc[i].material_id)
            slices_NdSiRu = backend.structure2SLICES(original_structure)
            reconstructed_structure, final_energy_per_atom = backend.SLICES2structure(slices_NdSiRu)
            is_okay = compare_structures(original_structure, reconstructed_structure)
            if not is_okay:
                # reconstructed_structure, final_energy_per_atom = backend.relax(reconstructed_structure)
                reconstructed_structure, final_energy_per_atom = run_with_memory_limit(get_relax, slices=slices_NdSiRu)
                is_okay = compare_structures(original_structure, reconstructed_structure)
            # if is_okay: correct += 1
            end_time = time.time()
            # data.append(f"{' ' if is_okay else 'x'} {i}: {df.iloc[i]['material_id']} --- {end_time - start_time}")
            queue.put((worker_id, f"{' ' if is_okay else 'x'} {idx}: {material_id} --- {end_time - start_time}"))
        except Exception as e:
            # print(e)
            # error += 1
            end_time = time.time()
            # data.append(f"E {i}: {df.iloc[i]['material_id']} --- {end_time - start_time}")
            queue.put((worker_id, f"E {idx}: {material_id} --- {end_time - start_time}"))
        finally:
            i += 1
            # if i % gc_each == 0:
            #     gc.collect()
            # if i % (gc_each * 2) == 0:
            #     backend = SLICES()
            # print(mem.percent, mem.available / mem.total, mem.available / mem.total < (1 - ram_threshold), 1 - ram_threshold)
            # if mem.available / mem.total < (1 - ram_threshold):
            # print(mem.percent, ram_threshold)
            if psutil.virtual_memory().percent > ram_threshold:
                free_queue.put((worker_id, rows[i:]))
                return

    queue.put((worker_id, None))

def run_split_parallel(df_func, start_from=0, append_previous_result=True, jobs=2, lost_file=None):
    df, split = df_func()
    correct = 0
    overall = 0
    error = 0
    initial_size = df.shape[0]
    file_to_save = f"results-{split}.txt" if lost_file is None else f"results-{lost_file}"
    to_gc = 200
    if lost_file is not None:
        with open(lost_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if len(lines) > 0:
                if not lines[-1]:
                    lines = lines[:-1]
                lines = [line[:-1] for line in lines]
                num_lines = len(lines)
                initial_size = num_lines
                df = df[df['material_id'].isin(lines)]
    if (append_previous_result or start_from > 0) and os.path.exists(file_to_save):
        with open(file_to_save, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if len(lines) > 0:
                if not lines[-1]:
                    lines = lines[:-1]
                num_lines = len(lines)
                correct = sum(1 for l in lines if l[0] == ' ')
                overall = num_lines
                error = sum(1 for l in lines if l[0] == 'E')
                start_from = num_lines
                to_remove_ids = get_materail_ids(file_to_save)
                df = df[~df['material_id'].isin(to_remove_ids)]
    data = []
    # backend = SLICES(relax_model="chgnet", steps=100)
    # backend = SLICES()
    
    batch_size = df.shape[0] // jobs
    queue = Queue()

    freed_processes_queue = Queue()

    processes = []
    for i in range(jobs):
        start = i * batch_size
        end = None if i == jobs-1 else (i+1) * batch_size
        batch = df.iloc[start:end]
        p = Process(target=job, args=(batch, queue, i, freed_processes_queue))
        processes.append(p)
        p.start()

    i = start_from
    finished_workers = 0
    with tqdm(total=initial_size, initial=start_from, smoothing=0.05) as pbar:
        while finished_workers < jobs:
            worker_id, string = queue.get()
            if string is not None:
                if string[0] == 'E':
                    error += 1
                elif string[0] == ' ':
                    correct += 1

                overall += 1
                data.append(string)

                pbar.set_postfix(correct=f"{correct}/{overall}", accuracy=f"{correct/overall:.2%}", error=f"{error}")
                pbar.update(1)
                i += 1

                if i % 10 == 9:
                    with open(file_to_save, "a", encoding="utf-8") as f:
                        f.writelines(line + "\n" for line in data)
                    data = []

                if i % to_gc == 0:
                    gc.collect()

                try:
                    freed_process, left_job = freed_processes_queue.get_nowait()
                    processes[freed_process] = Process(target=job, args=(left_job, queue, freed_process, freed_processes_queue))
                    processes[freed_process].start()
                except Empty:
                    pass
            else:
                finished_workers += 1
        with open(file_to_save, "a", encoding="utf-8") as f:
            f.writelines(line + "\n" for line in data)
    for p in processes:
        p.join()

E0000 00:00:1748539309.471214   21597 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748539309.481104   21597 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748539309.506046   21597 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748539309.506071   21597 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748539309.506073   21597 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748539309.506075   21597 computation_placer.cc:177] computation placer already registered. Please check linka

Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


In [8]:
# run_split_parallel(None, jobs=12)

In [ ]:
#Find poteryashki
run_split_parallel(get_mp20_split(None), jobs=8, lost_file="lost.txt")

  0%|          | 0/21 [00:00<?, ?it/s]/home/shtnp/Crystal-generator-using-SLICES/.venv/lib/python3.10/site-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitive cell, please set primitive=True explicitly.
  warnings.warn(
  5%|▍         | 1/21 [00:06<00:13,  1.49it/s, accuracy=0.00%, correct=0/2, error=2]/home/shtnp/Crystal-generator-using-SLICES/.venv/lib/python3.10/site-packages/pymatgen/io/cif.py:1186: UserWarning: The default value of primitive was changed from True to False in https://github.com/materialsproject/pymatgen/pull/3419. CifParser now returns the cell in the CIF file as is. If you want the primitive cell, please set primitive=True explicitly.
  warnings.warn(
 38%|███▊      | 8/21 [00:13<00:21,  1.67s/it, accuracy=62.50%, correct=5/8, error=3]/home/shtnp/Crystal-generator-using-S

In [10]:
# run_split_parallel("test", jobs=4)

In [11]:
# run_split_parallel("test")

In [12]:
import pandas as pd

df = pd.DataFrame({'material_id': ['a', 'b', 'c', 'd', 'e', 'f', 'g'], 'value': range(7)})
ids = ['b', 'e']
filtered_df = df[~df['material_id'].isin(ids)]

# chunk size 2
chunks = [filtered_df.iloc[i:i + 2] for i in range(0, len(filtered_df), 2)]
for i, chunk in enumerate(chunks):
    print(f'Chunk {i}:\n', chunk, '\n')

Chunk 0:
   material_id  value
0           a      0
2           c      2 

Chunk 1:
   material_id  value
3           d      3
5           f      5 

Chunk 2:
   material_id  value
6           g      6 



In [13]:
filtered_df.shape[0], len(filtered_df)

(5, 5)